<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">

<a href="https://colab.research.google.com/github/yhilpisch/algocolab/blob/main/notebooks/02_deep_learning_gpu_trading.ipynb"
target="_blank"><img
src="https://colab.research.google.com/assets/colab-badge.svg"
alt="Open In Colab"/></a>


# Algorithmic Trading with Python & Google Colab

## Session 2 — Teaching a Neural Network to Trade

### GPU Deep Learning and Honest Evaluation

The Python Quants GmbH | https://tpq.io<br>
© Dr. Yves J. Hilpisch | https://hilpisch.com

The live experiment follows one narrow, auditable path: reuse the Session 1
sample contract, fit a compact PyTorch classifier, select a trading threshold
on validation data, open the untouched test set once, and persist the complete
inference contract for Session 3.


## 1. Reconnect to the Active Drive Run

Session 1 writes an active-run pointer beside the immutable run directories.
This notebook reads that pointer and never selects the newest directory
implicitly. In Colab, Drive is mounted at
`/content/drive/MyDrive/algo`; locally the same files are available below the
synced macOS path.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True  # Use the Colab-specific workspace path
except ImportError:
    IN_COLAB = False  # Keep the notebook runnable outside Colab
def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]  # Check common local roots
    for candidate in candidates:
        if (candidate / 'src').is_dir():
            if (candidate / 'data' / 'eod_data.csv').is_file():
                return candidate
    raise FileNotFoundError(
        'Could not locate the companion repository root.'
    )
if IN_COLAB:
    drive.mount('/content/drive')  # Make persistent runs available
    PROJECT_ROOT = Path('/content/algocolab')  # Keep repository on local disk
    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                'git', 'clone', '--depth', '1',
                'https://github.com/yhilpisch/algocolab.git',
                str(PROJECT_ROOT),
            ],
            check=True,
        )
    RUNS_ROOT = Path('/content/drive/MyDrive/algo/runs')  # Persist artifacts
else:
    PROJECT_ROOT = find_project_root()  # Locate the local repository
    local_drive_runs = Path(
        '/Users/yves/Google Drive/My Drive/algo/runs'
    )
    RUNS_ROOT = Path(
        os.environ.get('WEBINAR_RUNS_ROOT', local_drive_runs)
    )
pointer_path = RUNS_ROOT / 'active_run.json'  # Read the Session 1 handoff
if not pointer_path.is_file():
    raise FileNotFoundError(
        f'Run pointer not found: {pointer_path}. '
        'Run Session 1 first.'
    )
pointer = json.loads(pointer_path.read_text(encoding='utf-8'))  # Parse handoff
RUN_ID = str(pointer.get('run_id', '')).strip()  # Extract immutable ID
if not RUN_ID:
    raise ValueError('The active run pointer has no run ID.')
PERSIST_RESULTS = os.environ.get(
    'WEBINAR_PERSIST_RESULTS',
    str(IN_COLAB),
).lower() in {'1', 'true', 'yes'}
sys.path.insert(0, str(PROJECT_ROOT))  # Import companion modules
print(f'Run: {RUN_ID}')

The next cell opens the immutable Session 1 run, recreates its
configuration, and imports the high-level Session 2 functions. It does not
fit a model or alter the stored Session 1 artifacts.


In [ ]:
from src.artifacts import RunBundle
from src.config import ExperimentConfig
from src.session2 import persist_session_two, run_session_two
bundle = RunBundle.open(  # Verify the inherited Session 1 bundle
    RUNS_ROOT,
    RUN_ID,
    required_session=1,
)
config = ExperimentConfig(**bundle.manifest['configuration'])  # Reuse config
session_one_metrics = bundle.path / 'session_1/strategy_metrics.csv'
print(f'Session 1 verified: {session_one_metrics}')
print(config)

The next cell confirms the data-generating question before we
construct the chronological feature samples. It creates no new run and does
not change the inherited Session 1 artifacts.


## 2. What the Network Is—and Is Not—Testing

The classifier estimates the probability of a positive next-day EUR/USD
return from lagged returns, rolling volatility, and rolling momentum. It is a
non-linear alternative inside the prediction branch of algorithmic trading;
it says nothing about the viability of market making, execution, arbitrage,
or other non-directional strategies.


## 3. Build Features and Preserve Time Order

The feature matrix is built from information available before the target
return. Splitting remains chronological: the model never trains on future
observations.


In [ ]:
import pandas as pd
from src.data import create_lagged_features, load_eod_data
prices = load_eod_data(  # Load the frozen EUR/USD price snapshot
    PROJECT_ROOT / 'data' / 'eod_data.csv',
    symbol=config.symbol,
)
prices = prices.loc[config.data_start:config.data_end]  # Apply experiment dates
features, target_return, target_direction = create_lagged_features(
    prices,
    lags=config.target_lags,
)  # Create pre-return features and next-day targets
train_end = int(len(features) * config.train_ratio)  # End training window
validation_end = int(
    len(features) * (config.train_ratio + config.validation_ratio)
)  # End validation window
x_train = features.iloc[:train_end]  # Earliest observations train the model
x_validation = features.iloc[train_end:validation_end]  # Tune later data
x_test = features.iloc[validation_end:]  # Reserve untouched test data
y_train = target_direction.iloc[:train_end]  # Training directions
y_validation = target_direction.iloc[train_end:validation_end]  # Validation
y_test = target_direction.iloc[validation_end:]  # Test directions
print(features.columns.tolist())
print(x_train.shape, x_validation.shape, x_test.shape)
x_train.head()

Scaling parameters belong to the training sample only. The same
training mean and scale are then applied to validation and test features.
They are part of the fitted inference contract and must be stored with the
model weights for later predictions.


In [ ]:
train_mean = x_train.mean()  # Fit centre on training data only
train_scale = x_train.std(ddof=0).replace(0.0, 1.0)  # Avoid zero division
x_train_scaled = (x_train - train_mean) / train_scale  # Scale training data
x_validation_scaled = (x_validation - train_mean) / train_scale  # Reuse scale
x_test_scaled = (x_test - train_mean) / train_scale  # Preserve test isolation
print('Training mean:')
print(train_mean.round(6))
print('Training scale:')
print(train_scale.round(6))

### Inspect the Custom Model Interface

The next cell imports the companion project's model interfaces, defines the
compact architecture, and creates a chronological training loader. Because
each lagged observation is already a time-ordered sample, the loader preserves
that order rather than shuffling it.
`ModelConfig`, `TradingDataset`, and `build_model` are project code, not
PyTorch APIs; `build_model` is the high-level factory for `TradingDNN`.


In [ ]:
import torch
from torch.utils.data import DataLoader
from src.models import ModelConfig, TradingDataset, build_model
model_config = ModelConfig(  # Define the compact network architecture
    input_dim=x_train_scaled.shape[1],
    hidden_units=(64, 32),
    dropout_rate=0.2,
)
model = build_model(model_config)  # Preview the custom TradingDNN architecture
train_loader = DataLoader(
    TradingDataset(x_train_scaled, y_train),  # Use training features/targets
    batch_size=64,
    shuffle=False,  # Preserve chronological order in the time series
)
parameter_count = sum(  # Count parameters eligible for training
    parameter.numel() for parameter in model.parameters()
)
print(model)  # Display the custom model architecture
print(f'Trainable parameters: {parameter_count:,}')

The custom `TradingDNN` maps the feature vector through two hidden
layers and emits a raw logit $z$. For readability, the equations omit batch
normalization and dropout, although the configured model includes both:

$$
\begin{aligned}
h_1&=\operatorname{ReLU}(W_1x+b_1),\\
h_2&=\operatorname{ReLU}(W_2h_1+b_2),\\
z&=W_3h_2+b_3,\qquad
\hat p=\sigma(z)=\frac{1}{1+e^{-z}}.
\end{aligned}
$$

The custom trainer passes raw $z$ to `BCEWithLogitsLoss`, which applies the
Sigmoid internally and calculates binary cross-entropy stably. The probability
$\hat p$ is used for inference and thresholding; the loss does not directly
optimize return, drawdown, turnover, or Sharpe ratio.


### Inspect the Assigned Runtime and Device

The next cell reports the PyTorch version, the assigned Colab runtime's
execution device, and the CUDA GPU name when one is available. It also remains
runnable on Apple MPS or CPU outside Colab.


In [ ]:
import time
import torch
from src.models import get_device
device = get_device()  # Select CUDA, MPS, or CPU for PyTorch operations
print(f'PyTorch: {torch.__version__}')  # Report the installed PyTorch version
print(f'Device: {device}')  # Report the selected execution device
print(f'Accelerator: {device.type}')  # Report the device category
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))  # Identify the assigned CUDA GPU

### Colab Runtime: Capacity Is Variable

Colab can assign CPU, GPU, or TPU runtimes. GPU sessions may use models such
as T4, L4, A100, G4, or H100, subject to plan, region, and current
availability. Inspect the assigned Colab runtime and its device, then measure
this workload rather than expecting a particular speed-up. This PyTorch
workflow uses CPU/CUDA in Colab; it does not use the TPU runtime.

Memory and precision support affect feasible batch size and training time, but
not the chronological split, validation-only threshold selection, or untouched
test rule.


### Tensor Placement: CPU, CUDA, and MPS

The full dataset starts in ordinary CPU memory. With CUDA, PyTorch places the
model state and each current mini-batch in the NVIDIA GPU's separate VRAM.
With Apple MPS, CPU and GPU share unified memory, but placement on `mps` still
selects the MPS execution backend. Model, features, and targets must share the
selected device. CUDA batch size is limited by VRAM; MPS workloads are limited
by available unified memory.

The next cell times a small tensor workload on the assigned device.


In [ ]:
def synchronize_device() -> None:
    if device.type == 'cuda':
        torch.cuda.synchronize()  # Finish queued CUDA work before timing
    elif device.type == 'mps':
        torch.mps.synchronize()  # Finish queued MPS work before timing
matrix = torch.randn(1024, 1024, device=device)  # Allocate a device tensor
synchronize_device()  # Exclude earlier accelerator work from the timing
started = time.perf_counter()  # Start the wall-clock timer
for _ in range(10):
    matrix @ matrix  # Perform one matrix multiplication
synchronize_device()  # Wait for queued accelerator work to finish
elapsed = (time.perf_counter() - started) / 10  # Average ten operations
print(f'1,024 x 1,024 matmul: {elapsed:.4f}s')  # Report mean elapsed time

## 4. Train a 20-Member Probability Ensemble

All scaling parameters are fitted on the training sample only. The
chronological validation and test partitions remain later in time. The
canonical runner trains the same compact architecture from 20 predeclared
initialization seeds. It averages their sigmoid probabilities before choosing
one trading threshold and records the complete evaluation contract.


In [ ]:
started = time.perf_counter()  # Start the end-to-end training timer
results = run_session_two(  # Train and evaluate the canonical experiment
    PROJECT_ROOT / 'data' / 'eod_data.csv',
    config,
    epochs=20,
    hidden_units=(64, 32),
    dropout_rate=0.2,
    ensemble_members=config.ensemble_members,
    device=device,
)
elapsed = time.perf_counter() - started  # Measure the complete run time
parameter_count = sum(  # Count the trained model's parameters
    parameter.numel() for parameter in results.models[0].parameters()
)
print(f'Training time: {elapsed:.2f}s')  # Report end-to-end training time
print(f'Ensemble members: {len(results.models)}')  # Report ensemble size
print(f'Parameters per member: {parameter_count:,}')  # Report model size
print(f'Seeds: {results.seeds}')  # Report the predeclared seed sequence
print(results.models[0])  # Display the shared member architecture
results.history.tail()  # Inspect final training/validation losses

The next cell plots mean training and validation losses across
the ensemble members. Read their relationship rather than training loss alone:
each member retains the state with its lowest validation loss, not necessarily
its final epoch.


In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')  # Apply the house plotting style
ax = results.history[['train_loss', 'val_loss']].plot(
    figsize=(9, 4),
    color=['#002D5A', '#2F80ED'],
)
ax.set(title='Training and validation loss', xlabel='Epoch')  # Label curves
ax.grid(alpha=0.35)  # Add a light reading grid
plt.show()  # Render the learning-curve diagnostic

## 5. Select the Trading Rule on Validation Data

For a symmetric threshold $\theta$, the position is long when the ensemble's
mean probability is greater than $\theta$, short when it is less than
$1-\theta$, and flat inside the deadband. We compare $\theta\in\{0.50,
0.52,0.55\}$: for example, $\theta=0.55$ means long above 55%, short below
45%, and no position between 45% and 55%. Thresholds are ranked only by
validation Sharpe after the canonical 0.5 basis point one-way cost. The test
sample is not used for this choice.

The selected active fraction shows how often the DNN holds a non-zero position.
At $\theta=0.50$, it is effectively always long or short; thresholds above
0.50 introduce flat days.


In [ ]:
threshold_columns = [  # Select the validation comparison columns
    'threshold',
    'active_fraction',
    'net_annual_return',
    'net_sharpe',
    'maximum_drawdown',
    'turnover_units',
]
results.threshold_results[threshold_columns]  # Display candidate thresholds

The next cell identifies the validation-winning threshold and its
active fraction. This makes clear whether the selected DNN rule actually uses
a confidence deadband.


In [ ]:
selected_row = results.threshold_results.loc[
    results.threshold_results['threshold'].eq(results.threshold)
].iloc[0]  # Read the validation-winning threshold row
validation_active = selected_row['active_fraction']  # Store selected exposure
print(f'Validation-selected threshold: {results.threshold:.2f}')  # Report theta
print(f'Validation active fraction: {validation_active:.1%}')  # Report exposure

## 6. Inspect Seed Dispersion and Open the Test Set Once

Individual members can follow materially different optimization paths. The
next table reports each member using a threshold chosen from that member's
validation probabilities. These rows diagnose seed sensitivity; they do not
select a winning member for deployment.


In [ ]:
seed_test = results.seed_metrics.query(  # Keep member test rows
    "sample == 'test'"
)
seed_summary = seed_test[  # Summarize the predeclared seed distribution
    ['net_annual_return', 'net_sharpe', 'maximum_drawdown']
].agg(['mean', 'std', 'median', 'min', 'max'])
seed_summary  # Display dispersion without selecting a lucky member

The deployable strategy is the probability ensemble, not the
mean of separately backtested returns. The same test dates and cost convention
are applied to the ensemble DNN, OLS, momentum, random, and buy-and-hold
baselines. The ensemble threshold is selected once from its averaged
validation probabilities and then applied unchanged to averaged test
probabilities.

The test active fraction shows whether the selected rule creates flat days.
Negative findings remain visible: an ensemble reduces initialization
sensitivity but does not manufacture stable alpha.


In [ ]:
test_metrics = results.strategy_metrics.query(
    "sample == 'test'"
)  # Keep only untouched test-period rows
test_predictions = results.predictions.query(
    "sample == 'test'"
)  # Keep DNN positions on untouched test dates
dnn_active_fraction = test_predictions['dnn_position'].ne(0.0).mean()
print(f'DNN test active fraction: {dnn_active_fraction:.1%}')  # Report exposure
test_metrics.set_index('strategy')[
    [
        'net_annual_return',
        'net_volatility',
        'net_sharpe',
        'maximum_drawdown',
        'turnover_units',
    ]
]  # Compare all strategy results after costs

> **Research deepening beyond the live skeleton**
>
> - use walk-forward retraining and time-window diagnostics;
> - correct for repeated model and feature searches;
> - enrich costs with slippage, financing, market impact, and capacity;
> - test alternative targets, horizons, architectures, and calibration.
>
> These extensions strengthen inference; they must not be used to search the
> untouched test sample for a better story.


## 7. Persist the Exact Inference Contract

The ensemble checkpoint contains all member weights and seeds, the exact
architecture—including dropout placement—the aggregation rule, feature order,
training scaler, selected threshold, and run ID. Session 3 refuses to proceed
without this completed, checksummed bundle.


In [ ]:
if PERSIST_RESULTS:
    code_commit = subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    persist_session_two(
        bundle,
        results,
        code_commit=code_commit,
    )
    print(f'Session 2 persisted: {bundle.path}')
else:
    message = (
        'Persistence disabled; set WEBINAR_PERSIST_RESULTS=true to enable.'
    )
    print(message)  # Explain how to enable artifact persistence

## Session 2 Takeaway

Model capacity can discover non-linear patterns, but the measured test result
decides whether those patterns generalize economically. Averaging predictions
over predeclared seeds reduces initialization sensitivity without selecting a
lucky run. The durable output is a validated ensemble inference contract ready
for the paper-trading simulation in Session 3.


---

<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">
